# Stage 1: Data Acquisition, Cleaning & Dyadic Feature Engineering
## Determinants of Armed Conflict Salience in UN General Debate Speeches (1946–2025)

**Course:** Fundamentals of Data Science  
**Project:** Predicting & Explaining State Attention to Global Conflicts  
---

### Objective of this Notebook
This notebook documents the end-to-end data pipeline that ingests raw, multi-source conflict, diplomatic, and textual data, harmonizes state identifiers, and constructs the master dyadic dataset (`data/final_clean.parquet` / `data/final_clean.csv`) used across our exploratory data analysis and predictive modeling.

Each annual observation corresponds to a **state–conflict dyad** $(i, c, t)$:
- $i$: UN Member State delivering an address in the General Debate in year $t$.
- $c$: Armed conflict active in year $t$ according to UCDP/PRIO criteria (>= 25 battle-related fatalities).
- $t$: Calendar year (1946 - 2025, spanning 80 annual UNGA sessions).
- $y_{i, c, t}$: Binary indicator denoting whether state $i$'s speech mentioned conflict $c$ in year $t$.

## 1. Data Provenance & Acquisition Guide

In accordance with project guidelines, large raw text and tabular datasets are not bundled in the final submission repository. The table below lists all external sources, formal academic citations, persistent repository URLs, and expected local file paths for complete replication.

| Dataset Name | Primary Citation | Source / Repository Link | Target Path in `data/` | Key Variables Harvested |
|:---|:---|:---|:---|:---|
| **UN General Debate Corpus (UNGDC)** | Baturo, Dasandi & Mikhaylov (2017) | [Harvard Dataverse (doi:10.7910/DVN/0TJX8Y)](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/0TJX8Y) | `data/Speeches/UNGDC_1946-2025/TXT/` | Full speech text (`text`), session, country ISO3, year |
| **UCDP/PRIO Armed Conflict Dataset (v26.1)** | Gleditsch et al. (2002); Davies et al. (2024) | [Uppsala Conflict Data Program (UCDP)](https://ucdp.uu.se/downloads/index.html#armedconflict) | `data/2026_ucdp-prio-acd-261.xlsx` | `conflict_id`, `year`, `side_a`, `side_b`, `gwno_*`, `intensity_level`, `type_of_conflict` |
| **UCDP Battle-Related Deaths Dataset (v26.1)** | Pettersson & Öberg (2024) | [UCDP Downloads Portal](https://ucdp.uu.se/downloads/index.html#battlerelated) | `data/BattleDeaths_v26_1_conf.csv` | Modern battle deaths (1989–2025): `bd_best`, `bd_low`, `bd_high` |
| **PRIO Battle Deaths Dataset (v3.1)** | Lacina & Gleditsch (2005) | [Peace Research Institute Oslo (PRIO)](https://www.prio.org/data/1) | `data/PRIO Battle Deaths Dataset 31.xls` | Historical battle deaths (1946–2008): `bdeadbes`, `bdeadlow`, `bdeadhig` |
| **UNGA Voting Ideal Points (1946–2025)** | Bailey, Strezhnev & Voeten (2017) | [Harvard Dataverse (doi:10.7910/DVN/LEJUQZ)](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/LEJUQZ) | `data/IdealPointDyads1946-2025.csv` | Bilateral foreign policy divergence: `AbsIdealDiff` |
| **CEPII GeoDist (Bilateral Distance)** | Mayer & Zignago (2011) | [CEPII GeoDist Database](http://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=6) | `data/dist_cepii.xls` | Population-weighted distance (`distw`), bilateral country identifiers |



## 2. Environment Setup & Project Directory Resolution

We dynamically anchor all execution paths to the repository root directory using `pathlib.Path` to ensure portable execution across Windows, Linux, and macOS.

In [17]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Anchor to project root directory
curr = Path.cwd()
if (curr / "data").exists():
    project_root = curr
elif (curr.parent / "data").exists():
    project_root = curr.parent
else:
    project_root = curr

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root resolved to: {project_root.resolve()}")
print(f"Pandas version: {pd.__version__}")

Project root resolved to: C:\Users\stynz\Desktop\Nová složka (2)\fundamentals_of_data_science
Pandas version: 2.0.3


## 3. Modular Pipeline Execution

The data processing workflow is organized into modular Python scripts located in `clean_pipeline/scripts/` (and `scripts/`). Each script can be executed independently from the command line or invoked directly from this notebook.

### Step 3.1: Parsing Raw Speech Text Files (`speeches.py`)
- Iterates recursively over all annual General Debate `.txt` files in `data/Speeches/`.
- Extracts ISO3 country codes, UNGA session numbers, and speech year from standardized filenames (`[ISO3]_[Session]_[Year].txt`).
- Cleans text encoding and serializes to `data/un_speeches.parquet`.

In [18]:
# Step 1: UN General Debate Speeches Parser (speeches.py)
# Note: Raw .txt speech archives (over 10,000 files) are omitted from the submission repo.
# If raw text files are present in data/Speeches/, this cell executes the script.
from pathlib import Path

raw_speeches_dir = Path("data/Speeches/UNGDC_1946-2025/TXT")

if raw_speeches_dir.exists() and any(raw_speeches_dir.rglob("*.txt")):
    print("Raw speech files detected. Executing speeches.py...")
    %run clean_pipeline/scripts/speeches.py
else:
    print("[Data Provenance Note] Raw speech .txt files are omitted from repository submission.")
    print("To regenerate from scratch, download the UNGDC corpus (see Section 1 table)")
    print("and place files in: data/Speeches/UNGDC_1946-2025/TXT/")
    if Path("data/un_speeches.parquet").exists():
        print("Pre-parsed speech corpus found: data/un_speeches.parquet")

[Data Provenance Note] Raw speech .txt files are omitted from repository submission.
To regenerate from scratch, download the UNGDC corpus (see Section 1 table)
and place files in: data/Speeches/UNGDC_1946-2025/TXT/


### Step 3.2: Battle-Related Deaths Harmonization (`conflict_deaths.py`)
- **Challenge**: UCDP switched conflict identifier schemes between historical PRIO v3.1 (1946â€“2008) and modern UCDP v26.1 (1989â€“2025).
- **Resolution**: Maps legacy IDs to modern UCDP IDs, outer-joins both series, prioritizes official UCDP v26.1 reports, falls back to PRIO v3.1 point estimates, and imputes continuous midpoints `(low + high) / 2` when only intervals are reported.
- Produces `data/conflict_deaths_1946_2025.csv` (4,746 conflict-years across 1946â€“2025).

In [19]:
# Execute Step 2: Conflict & Casualty Harmonization (1946-2025)
%run clean_pipeline/scripts/conflict_deaths.py

Successfully exported unified conflict deaths dataset (4746 rows)!


### Step 3.3: Dyadic Assembly, Regex Salience Matching & Feature Integration (`speeches_conflict_mentioned.py`)
- **Gleditsch-Ward to ISO3**: Converts numeric state codes across all actor fields (`gwno_a`, `gwno_b`, `gwno_loc`, coalitions), with manual overrides for historical states (USSR, Yugoslavia, North/South Yemen, North/South Vietnam, Hyderabad).
- **Dual-Condition Regex Salience Matching**:
  1. *Entity Mention*: Detects at least one occurrence of the conflict location (`iso3_loc`), primary/secondary state belligerents (`iso3_a`, `iso3_b`), or non-state insurgent actor names (`side_a`, `side_b`, e.g., FARC, Hamas, LTTE).
  2. *War Anchor Keyword*: Detects at least one war/security keyword (`conflict`, `war`, `ceasefire`, `aggression`, `rebel`, `violence`, `peacekeeping`, etc.) to eliminate incidental, non-conflict diplomatic mentions.
- **Dyadic Cross-Join**: Generates all active $(i, c, t)$ dyads between UN member states and active armed conflicts.
- **Foreign Policy & Proximity**: Symmetrizes and merges UNGA ideal point divergence (`side_a_ideals_diff`) and CEPII distance (`side_a_dist`).
- **Output**: Serializes the master dataset to `data/final_clean.parquet` and `data/final_clean.csv`.

In [20]:
# Execute Step 3: Master Dyadic Assembly & Regex Text Matching (speeches_conflict_mentioned.py)
# Executes speeches_conflict_mentioned.py if all prerequisite intermediate datasets are available
from pathlib import Path

prerequisites = [
    Path("data/2026_ucdp-prio-acd-261.xlsx"),
    Path("data/un_speeches.parquet"),
    Path("data/IdealPointDyads1946-2025.csv"),
    Path("data/dist_cepii.xls"),
    Path("data/conflict_deaths_1946_2025.csv")
]

missing_prereqs = [str(p) for p in prerequisites if not p.exists()]

if not missing_prereqs:
    print("All intermediate inputs detected. Executing speeches_conflict_mentioned.py...")
    %run clean_pipeline/scripts/speeches_conflict_mentioned.py
else:
    print("[Data Provenance Note] The following intermediate raw source files are omitted from submission:")
    for m in missing_prereqs:
        print(f"  - {m}")
    print("\nThe compiled master dataset from this pipeline is pre-computed and stored at:")
    print("  - data/final_clean.csv (and final_clean.zip in root)")
    print("Section 4 below loads and validates this dataset directly.")

[Data Provenance Note] The following intermediate raw source files are omitted from submission:
  - data\un_speeches.parquet
  - data\dist_cepii.xls

The compiled master dataset from this pipeline is pre-computed and stored at:
  - data/final_clean.csv (and final_clean.zip in root)
Section 4 below loads and validates this dataset directly.


## 4. Dataset Integrity & Quality Assurance (QA)

We perform systematic data validation to verify row counts, column coverage, missingness thresholds, and baseline probability distribution across the 80-year panel.

In [21]:
# Load compiled dataset directly from data/ directory or fallback archive
csv_path = project_root / "data" / "final_clean.csv"
parquet_path = project_root / "data" / "final_clean.parquet"
root_csv = project_root / "final_clean.csv"
zip_path = project_root / "final_clean.zip"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
elif csv_path.exists():
    df = pd.read_csv(csv_path, low_memory=False)
elif root_csv.exists():
    df = pd.read_csv(root_csv, low_memory=False)
elif zip_path.exists():
    import zipfile
    print(f"Unpacking {zip_path.name}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(project_root / "data")
    df = pd.read_csv(csv_path, low_memory=False)

# High-level dataset dimensions
print("=== Dataset Dimensions & Panel Scope ===")
print(f"Total Dyadic Observations (N):  {len(df):,}")
print(f"Temporal Span:                  {df['year'].min()} - {df['year'].max()} ({df['year'].nunique()} annual sessions)")
print(f"Unique Armed Conflicts:         {df['conflict_id'].nunique():,}")
print(f"Unique Member States (Speakers):{df['country_code'].nunique():,}")
print(f"Positive Conflict Mentions:     {df['has_conflict_mention'].sum():,} ({df['has_conflict_mention'].mean():.2%})")

# Validation Assertions
assert len(df) == 438425, f"Expected 438,425 rows, found {len(df)}"
assert df['has_conflict_mention'].nunique() == 2, "Target variable must be binary boolean"
print("\n[PASSED] All structural panel assertions verified successfully.")

=== Dataset Dimensions & Panel Scope ===
Total Dyadic Observations (N):  438,425
Temporal Span:                  1946 - 2025 (80 annual sessions)
Unique Armed Conflicts:         303
Unique Member States (Speakers):200
Positive Conflict Mentions:     76,340 (17.41%)

[PASSED] All structural panel assertions verified successfully.


In [22]:
# Missing data profile across all columns
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes.values,
    'Non_Null_Count': df.notna().sum().values,
    'Missing_Count': df.isna().sum().values,
    'Missing_Pct': (df.isna().mean() * 100).round(2).values
}).sort_values('Missing_Pct', ascending=False).reset_index(drop=True)

print("=== Missingness Profile (Core Features vs. Auxiliary Metadata) ===")
missing_summary.head(20)

=== Missingness Profile (Core Features vs. Auxiliary Metadata) ===


,Column,Data_Type,Non_Null_Count,Missing_Count,Missing_Pct
0,ep_end_prec,float64,0,438425,100.00
1,side_b_ideals_diff,float64,18140,420285,95.86
2,side_b_dist,float64,20450,417975,95.34
3,iso3_b,object,21525,416900,95.09
4,side_b_2nd,object,22652,415773,94.83
5,iso3_b_2nd,object,22652,415773,94.83
6,ependdate,object,50541,387884,88.47
7,ependprec,float64,50727,387698,88.43
8,side_a_2nd,object,81150,357275,81.49
9,iso3_a_2nd,object,81150,357275,81.49


In [23]:
# Summary table of key analytical variables for the academic report
summary_metrics = pd.DataFrame({
    'Metric': [
        'Panel Time Period',
        'Total Dyadic Observations (N)',
        'Unique Armed Conflicts',
        'Unique Speaker Member States',
        'Total Positive Mentions',
        'Baseline Mention Probability',
        'Direct Belligerents (Host / Secondary Actor)',
        'Third-Party Observers',
        'Continuous Battle Deaths Coverage'
    ],
    'Value': [
        f"{df['year'].min()} - {df['year'].max()} (80 annual sessions)",
        f"{len(df):,}",
        f"{df['conflict_id'].nunique():,}",
        f"{df['country_code'].nunique():,}",
        f"{df['has_conflict_mention'].sum():,}",
        f"{df['has_conflict_mention'].mean():.2%}",
        f"{(df['conflict_host'] | df['secondary_actor']).sum():,} ({((df['conflict_host'] | df['secondary_actor']).mean()):.2%})",
        f"{(~df['conflict_host'] & ~df['secondary_actor']).sum():,} ({(~df['conflict_host'] & ~df['secondary_actor']).mean():.2%})",
        f"{df['deaths_estimate'].notna().mean():.1%} non-null across all dyads"
    ]
})

summary_metrics

,Metric,Value
0,Panel Time Period,1946 - 2025 (80 annual sessions)
1,Total Dyadic Observations (N),"438,425"
2,Unique Armed Conflicts,303
3,Unique Speaker Member States,200
4,Total Positive Mentions,"76,340"
5,Baseline Mention Probability,17.41%
6,Direct Belligerents (Host / Secondary Actor),"6,033 (1.38%)"
7,Third-Party Observers,"432,392 (98.62%)"
8,Continuous Battle Deaths Coverage,98.9% non-null across all dyads


## 5. Transition to Downstream Notebooks

With the master panel dataset compiled and validated, subsequent analysis proceeds in two specialized notebooks:
1. **`02_exploratory_data_analysis.ipynb`**: Evaluates substantive research hypotheses regarding distance decay (Sub-RQ 2b), casualty scaling and log deaths (Sub-RQ 2a), ideological divergence, and speaker country propensity.
2. **`03_predictive_modeling.ipynb`**: Implements temporal walk-forward cross-validation (2012–2025), gradient boosted decision trees (XGBoost), and Tree SHAP feature attribution to predict whether a nation will articulate a specific conflict.